# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List record sets and their fields by @id
print('Available record sets and their fields:')
for record_set in dataset.record_sets:
    print(f"Record Set @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print(f"  Description: {getattr(record_set, 'description', '')}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    Field @id: {field.id} (name: {field.name}, type: {field.data_type})")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set into DataFrames
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Inspect the first record set's fields and some data
if record_set_ids:
    print(f'Record Set @id: {record_set_ids[0]} columns:')
    print(dataframes[record_set_ids[0]].columns.tolist())
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Let's identify a numeric field in the first record set for analysis
import numpy as np

selected_record_set_id = record_set_ids[0]
df = dataframes[selected_record_set_id]

# Automatically select the first float/integer field for demonstration
numeric_field_id = None
for field in dataset.record_sets[0].fields:
    if str(field.data_type).lower() in ['float', 'integer', 'number'] and field.id in df.columns:
        numeric_field_id = field.id
        break

if numeric_field_id is not None:
    print(f"Selected numeric field for analysis: {numeric_field_id}\n")
    # Convert to numeric if needed
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Set a simple threshold from observed data
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    col_mean = filtered_df[numeric_field_id].mean()
    col_std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - col_mean) / col_std
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to group by a categorical field
    group_field_id = None
    for field in dataset.record_sets[0].fields:
        if str(field.data_type).lower() in ['string', 'text'] and field.id != numeric_field_id and field.id in df.columns:
            group_field_id = field.id
            break

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"\nGrouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field found for analysis in the selected record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualize the distribution and relationships if possible
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8, 6))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset was successfully loaded using the Croissant schema and `mlcroissant`.
- We inspected the available record sets, their fields (by `@id`), and loaded them into Pandas DataFrames.
- Exploratory analysis was performed on a representative numeric field, including filtering, normalization, and grouping.
- Visualizations revealed the distribution of the selected numeric field and its relationship with a categorical attribute (if present).

Next steps could include feature engineering, applying ML models, or integrating with other bioinformatics pipelines.